# Logistic, RandomForest, XGB, LGBM + SVM

## 1. None 2. Boderline SMOTE  3. Class Weight 4. CTGAN

In [ ]:
# ============================================================
# 0. 라이브러리
# ============================================================

import pandas as pd
import numpy as np
import copy

from imblearn.over_sampling import BorderlineSMOTE
from ctgan import CTGAN

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

import warnings
warnings.filterwarnings("ignore")


# ============================================================
# 1. 설정값
# ============================================================

TRAIN_PATH = r'10,11,12번\train데이터\M19_도매_소매업_train.parquet'
TEST_PATH  = r'10,11,12번\test데이터\M19_도매_소매업_test.parquet'

TARGET_COL   = "부실라벨_ICR3년"
RANDOM_STATE = 42
THRESHOLD    = 0.5

YEAR_COL = "회계년도"
ID_COLS  = ["회사명", "사업자등록번호", "회계년도"]

FOLD_VAL_YEARS = [2016, 2017, 2018, 2019, 2020, 2021]
TRAIN_START    = 2012

PRIMARY_METRICS = ["F1", "Recall", "ROC_AUC", "Precision", "PR_AUC", "Accuracy"]


# ============================================================
# 2. 피처 파일
# ============================================================

feature_files = {
    # "top50_dedup43": r"13번.피처셀렉션\M19_도매_소매업\lasso_features_top50--43.csv",
    # "top55_dedup45": r"13번.피처셀렉션\M19_도매_소매업\lasso_features_top55--45.csv",
    # "top60_dedup49": r"13번.피처셀렉션\M19_도매_소매업\lasso_features_top60--49.csv",
    "top65_dedup52": r"13번.피처셀렉션\lasso_features_top65--52.csv",
}


# ============================================================
# 3. 데이터 로드
# ============================================================

train_full = pd.read_parquet(TRAIN_PATH)
test       = pd.read_parquet(TEST_PATH)

y_train_full = train_full[TARGET_COL]
y_test       = test[TARGET_COL]

train_ids = train_full[[c for c in ID_COLS if c in train_full.columns]].reset_index(drop=True)
test_ids  = test[[c for c in ID_COLS if c in test.columns]].reset_index(drop=True)

print("=" * 70)
print(f"Train shape: {train_full.shape} | Test shape: {test.shape}")
print("=" * 70)


# ============================================================
# 4. 피처 목록 로드 & 유효성 확인
# ============================================================

feature_map = {}

for feature_name, feature_path in feature_files.items():
    df_feat    = pd.read_csv(feature_path)
    col_key    = "feature" if "feature" in df_feat.columns else df_feat.columns[0]
    raw_features   = df_feat[col_key].tolist()
    valid_features = [f for f in raw_features if f in train_full.columns]
    feature_map[feature_name] = valid_features
    print(f"[{feature_name}]  파일 내 피처: {len(raw_features)}개  "
          f"→ 실제 사용(train 컬럼 교집합): {len(valid_features)}개")

print("=" * 70)


# ============================================================
# 5. pos_weight 계산
# ============================================================

pos_weight = (y_train_full == 0).sum() / (y_train_full == 1).sum()
print(f"XGBoost scale_pos_weight : {pos_weight:.4f}")
print(f"클래스 분포 → 정상(0): {(y_train_full==0).sum()}, 부실(1): {(y_train_full==1).sum()}")
print("=" * 70)


# ============================================================
# 6. 모델 군 정의
# ============================================================

models_base = {
    "LogisticRegression": LogisticRegression(
        penalty="l2", C=1.0, solver="lbfgs",
        max_iter=1000, random_state=RANDOM_STATE
    ),
    # "SVM": SVC(
    #     kernel="rbf", C=1.0, gamma="scale",
    #     probability=True,                    # predict_proba 사용을 위해 필수
    #     random_state=RANDOM_STATE
    # ),
    "RandomForest": RandomForestClassifier(
        n_estimators=300, max_depth=4, min_samples_leaf=5,
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="aucpr",                 # ★ f1 → aucpr 수정
        random_state=RANDOM_STATE, verbosity=0
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_STATE, verbose=-1
    ),
}

models_cw = {
    "LogisticRegression": LogisticRegression(
        penalty="l2", C=1.0, solver="lbfgs",
        max_iter=1000, random_state=RANDOM_STATE,
        class_weight="balanced"
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=300, max_depth=4, min_samples_leaf=5,
        random_state=RANDOM_STATE, n_jobs=-1,
        class_weight="balanced"
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="aucpr",
        random_state=RANDOM_STATE, verbosity=0,
        scale_pos_weight=pos_weight
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_STATE, verbose=-1,
        class_weight="balanced"
    ),
}


# ============================================================
# 7. 오버샘플링 함수
# ============================================================

def apply_none(X_train, y_train, ratio=None):
    return X_train.copy(), y_train.copy()


def apply_borderline_smote(X_train, y_train, ratio):
    smote = BorderlineSMOTE(
        sampling_strategy=ratio,
        random_state=RANDOM_STATE,
        kind="borderline-1"
    )
    X_res, y_res = smote.fit_resample(X_train, y_train)
    return (
        pd.DataFrame(X_res, columns=X_train.columns),
        pd.Series(y_res, name=TARGET_COL)
    )


def apply_ctgan(X_train, y_train, ratio):
    y_train = y_train.reset_index(drop=True)
    X_train = X_train.reset_index(drop=True)

    n_majority  = (y_train == 0).sum()
    n_minority  = (y_train == 1).sum()
    n_target    = int(n_majority * ratio)
    n_synthetic = max(n_target - n_minority, 0)

    if n_synthetic == 0:
        return X_train.copy(), y_train.copy()

    X_minority = X_train[y_train == 1].reset_index(drop=True)

    ctgan = CTGAN(epochs=300, verbose=False)  # epochs 300으로 증가
    ctgan.fit(X_minority)
    X_syn = ctgan.sample(n_synthetic)

    # ★ 핵심 수정: 컬럼 순서를 X_train 기준으로 명시적 정렬
    X_syn = X_syn[X_train.columns].reset_index(drop=True)

    # ★ 추가: CTGAN 합성 데이터 클리핑 (훈련 데이터 범위로 제한)
    for col in X_train.columns:
        col_min = X_train[col].min()
        col_max = X_train[col].max()
        X_syn[col] = X_syn[col].clip(col_min, col_max)

    X_res = pd.concat([X_train, X_syn], ignore_index=True)
    y_res = pd.concat([
        y_train,
        pd.Series([1] * n_synthetic, name=TARGET_COL)
    ], ignore_index=True)

    return X_res, y_res


# ============================================================
# 8. method_configs
# ============================================================

method_configs = {
    # "None": (
    #     apply_none,
    #     [None],
    #     models_base
    # ),
    # "ClassWeight": (
    #     apply_none,
    #     [None],
    #     models_cw
    # ),
    # "BorderlineSMOTE": (
    #     apply_borderline_smote,
    #     [0.1, 0.2],          # 0.03 불균형 기준 보수적 탐색
    #     models_base
    # ),
    "CTGAN": (
        apply_ctgan,
        [0.2],          # BorderlineSMOTE와 동일 기준으로 비교
        models_base
    ),
}                             # ★ 닫는 중괄호 (기존 코드에서 누락되어 있었음)


# ============================================================
# 9. 평가 지표 계산 헬퍼
# ============================================================

def calc_metrics(y_true, y_prob, threshold=THRESHOLD):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "F1"        : f1_score(y_true, y_pred, zero_division=0),
        "Recall"    : recall_score(y_true, y_pred, zero_division=0),
        "ROC_AUC"   : roc_auc_score(y_true, y_prob),
        "Precision" : precision_score(y_true, y_pred, zero_division=0),
        "PR_AUC"    : average_precision_score(y_true, y_prob),
        "Accuracy"  : accuracy_score(y_true, y_pred),
    }


# ============================================================
# 10. Expanding Window CV 인덱스 생성
# ============================================================

def make_expanding_folds(df, year_col, fold_val_years, train_start):
    folds = []
    for val_year in fold_val_years:
        train_idx = df.index[
            (df[year_col] >= train_start) & (df[year_col] < val_year)
        ]
        val_idx = df.index[df[year_col] == val_year]
        if len(train_idx) > 0 and len(val_idx) > 0:
            folds.append((train_idx, val_idx))
        else:
            print(f"  [경고] val_year={val_year} fold 생성 불가 (데이터 없음)")
    return folds


# ============================================================
# 11. 메인 실험 루프
# ============================================================

cv_results   = []
test_results = []

for feature_name, use_features in feature_map.items():

    feature_file_name = feature_files[feature_name].split("\\")[-1]
    n_features        = len(use_features)

    print(f"\n{'='*70}")
    print(f"[Feature Set: {feature_name}]  파일: {feature_file_name}  |  피처 수: {n_features}개")
    print(f"{'='*70}")

    folds = make_expanding_folds(train_full, YEAR_COL, FOLD_VAL_YEARS, TRAIN_START)
    print(f"  생성된 fold 수: {len(folds)}")

    for method_name, (oversample_fn, ratios, models_dict) in method_configs.items():

        print(f"\n  {'='*60}")
        print(f"  방식: {method_name}")

        for ratio in ratios:
            ratio_label = ratio if ratio is not None else "-"
            print(f"\n    ratio = {ratio_label}")

            # ── (A) Expanding Window CV ───────────────────────────
            for fold_idx, (train_idx, val_idx) in enumerate(folds, start=1):

                X_fold_train = train_full.loc[train_idx, use_features]
                y_fold_train = train_full.loc[train_idx, TARGET_COL]
                X_fold_val   = train_full.loc[val_idx,   use_features]
                y_fold_val   = train_full.loc[val_idx,   TARGET_COL]
                val_year     = FOLD_VAL_YEARS[fold_idx - 1]

                imputer      = SimpleImputer(strategy="median")
                X_fold_train = pd.DataFrame(
                    imputer.fit_transform(X_fold_train), columns=use_features
                )
                X_fold_val_imp = pd.DataFrame(
                    imputer.transform(X_fold_val), columns=use_features
                )

                X_res, y_res = oversample_fn(X_fold_train, y_fold_train, ratio)

                n0 = (y_res == 0).sum()
                n1 = (y_res == 1).sum()

                # ── LR / SVM용 스케일링 ───────────────────────────
                scaler       = StandardScaler()
                X_res_scaled = pd.DataFrame(
                    scaler.fit_transform(X_res), columns=use_features
                )
                X_val_scaled = pd.DataFrame(
                    scaler.transform(X_fold_val_imp), columns=use_features
                )

                for model_name, model in models_dict.items():
                    m = copy.deepcopy(model)

                    # ★ SVM 추가 — LR과 동일하게 스케일링 데이터 사용
                    if model_name in ("LogisticRegression", "SVM"):
                        m.fit(X_res_scaled, y_res)
                        y_prob_val = m.predict_proba(X_val_scaled)[:, 1]
                    else:
                        m.fit(X_res, y_res)
                        y_prob_val = m.predict_proba(X_fold_val_imp)[:, 1]

                    val_metrics = calc_metrics(y_fold_val, y_prob_val)

                    cv_results.append({
                        "FeatureSet"  : feature_name,
                        "FeatureFile" : feature_file_name,
                        "N_Features"  : n_features,
                        "Method"      : method_name,
                        "SMOTE_Ratio" : ratio_label,
                        "Model"       : model_name,
                        "Fold"        : fold_idx,
                        "Val_Year"    : val_year,
                        "Train_N0"    : n0,
                        "Train_N1"    : n1,
                        **{f"Val_{k}": v for k, v in val_metrics.items()},
                    })

            # ── (B) Test 평가: 전체 train으로 재학습 ─────────────
            X_train_all = train_full[use_features]
            y_train_all = train_full[TARGET_COL]

            imputer_final = SimpleImputer(strategy="median")
            X_train_all   = pd.DataFrame(
                imputer_final.fit_transform(X_train_all), columns=use_features
            )
            X_test_imp    = pd.DataFrame(
                imputer_final.transform(test[use_features]), columns=use_features
            )

            X_tr_res, y_tr_res = oversample_fn(X_train_all, y_train_all, ratio)

            scaler_final    = StandardScaler()
            X_tr_res_scaled = pd.DataFrame(
                scaler_final.fit_transform(X_tr_res), columns=use_features
            )
            X_test_scaled   = pd.DataFrame(
                scaler_final.transform(X_test_imp), columns=use_features
            )

            for model_name, model in models_dict.items():
                m = copy.deepcopy(model)

                if model_name in ("LogisticRegression", "SVM"):
                    m.fit(X_tr_res_scaled, y_tr_res)
                    y_prob_test = m.predict_proba(X_test_scaled)[:, 1]
                else:
                    m.fit(X_tr_res, y_tr_res)
                    y_prob_test = m.predict_proba(X_test_imp)[:, 1]

                test_metrics = calc_metrics(y_test, y_prob_test)

                test_results.append({
                    "FeatureSet"  : feature_name,
                    "FeatureFile" : feature_file_name,
                    "N_Features"  : n_features,
                    "Method"      : method_name,
                    "SMOTE_Ratio" : ratio_label,
                    "Model"       : model_name,
                    **{f"Test_{k}": v for k, v in test_metrics.items()},
                })

                print(
                    f"      [{model_name}]  "
                    f"Test F1={test_metrics['F1']:.4f} | "
                    f"Recall={test_metrics['Recall']:.4f} | "
                    f"PR_AUC={test_metrics['PR_AUC']:.4f}"
                )


# ============================================================
# 12. 결과 DataFrame 변환
# ============================================================

cv_df   = pd.DataFrame(cv_results)
test_df = pd.DataFrame(test_results)


# ============================================================
# 13. CV 평균 집계 + Test 결과 병합 → summary_df
# ============================================================

cv_agg = (
    cv_df
    .groupby(["FeatureSet", "FeatureFile", "N_Features", "Method", "SMOTE_Ratio", "Model"])
    [[f"Val_{m}" for m in PRIMARY_METRICS]]
    .mean()
    .reset_index()
)

summary_df = cv_agg.merge(
    test_df,
    on=["FeatureSet", "FeatureFile", "N_Features", "Method", "SMOTE_Ratio", "Model"],
    how="left"
)

for m in PRIMARY_METRICS:
    summary_df[f"Gap_{m}"] = summary_df[f"Val_{m}"] - summary_df[f"Test_{m}"]

print(f"\nsummary_df shape: {summary_df.shape}")
print(summary_df[["Method", "Model", "Val_F1", "Test_F1", "Gap_F1"]].to_string(index=False))


# ============================================================
# 14. 저장
# ============================================================

summary_df.to_csv(
    "Summary_CV_Test_모델별_파일별_불균형방식별++.csv",
    index=False, encoding="utf-8-sig"
)


Train shape: (28111, 256) | Test shape: (11797, 256)
[top65_dedup52]  파일 내 피처: 52개  → 실제 사용(train 컬럼 교집합): 52개
XGBoost scale_pos_weight : 25.6455
클래스 분포 → 정상(0): 27056, 부실(1): 1055

[Feature Set: top65_dedup52]  파일: lasso_features_top65--52.csv  |  피처 수: 52개
  생성된 fold 수: 6

  방식: CTGAN

    ratio = 0.2


BrokenProcessPool: A task has failed to un-serialize. Please ensure that the arguments of the function are all picklable.

In [6]:
import pandas as pd

# ============================================================
# 가중치 설정
# ============================================================
WEIGHTS = {
    "Test_Recall"  : 0.2,
    "Test_F1"      : 0.3,
    "Test_PR_AUC" : 0.5,
}

# ============================================================
# 데이터 로드
# ============================================================
df = pd.read_csv("Summary_CV_Test_모델별_파일별_불균형방식별.csv")

# ============================================================
# 가중 점수 계산
# ============================================================
df["Weighted_Score"] = sum(
    df[metric] * weight
    for metric, weight in WEIGHTS.items()
).round(4)

# ============================================================
# 출력 컬럼 정의
# ============================================================
ID_COLS     = ["FeatureSet", "FeatureFile", "N_Features", "Method", "SMOTE_Ratio", "Model"]
SCORE_COLS  = ["Weighted_Score", "Test_F1", "Test_Recall", "Test_Precision", "Test_PR_AUC","Test_ROC_AUC"]
GAP_COLS    = ["Gap_F1", "Gap_Recall", "Gap_Precision", "Gap_PR_AUC","Gap_ROC_AUC"]
DISPLAY_COLS = ID_COLS + SCORE_COLS + GAP_COLS

weight_str = " + ".join(f"{m}×{w}" for m, w in WEIGHTS.items())

# ============================================================
# 1) 전체 TOP 10
# ============================================================
top10 = (
    df[DISPLAY_COLS]
    .sort_values("Weighted_Score", ascending=False)
    .head(10)
    .reset_index(drop=True)
)
top10.index += 1

print("=" * 100)
print(f"★ 전체 최적 조합 TOP 10  ({weight_str})")
print(f"   가중치 합계: {sum(WEIGHTS.values()):.1f}")
print("=" * 100)
print(top10.to_string())

# ============================================================
# 2) 피처셋별 TOP 3
# ============================================================
print("\n" + "=" * 100)
print("★ 피처셋별 최적 조합 TOP 3")
print("=" * 100)

for feature_set, group in df.groupby("FeatureSet"):
    feature_file = group["FeatureFile"].iloc[0]
    n_feat       = group["N_Features"].iloc[0]

    top3 = (
        group[DISPLAY_COLS]
        .sort_values("Weighted_Score", ascending=False)
        .head(3)
        .reset_index(drop=True)
    )
    top3.index += 1

    print(f"\n  [{feature_set}]  파일: {feature_file}  |  피처 수: {n_feat}개")
    print(top3[["Method", "SMOTE_Ratio", "Model"] + SCORE_COLS + GAP_COLS].to_string())

# ============================================================
# 3) 피처셋별 1등 비교 요약
# ============================================================
print("\n" + "=" * 100)
print("★ 피처셋별 1등 요약 비교")
print("=" * 100)

best_per_set = (
    df.sort_values("Weighted_Score", ascending=False)
    .groupby("FeatureSet", sort=False)
    .first()
    .reset_index()
)[DISPLAY_COLS]

print(best_per_set.to_string(index=False))

top10.to_csv('15번. 우수모델 데이터/Top10 우수모델.csv')

★ 전체 최적 조합 TOP 10  (Test_Recall×0.2 + Test_F1×0.3 + Test_PR_AUC×0.5)
   가중치 합계: 1.0
       FeatureSet                   FeatureFile  N_Features       Method SMOTE_Ratio         Model  Weighted_Score   Test_F1  Test_Recall  Test_Precision  Test_PR_AUC  Test_ROC_AUC    Gap_F1  Gap_Recall  Gap_Precision  Gap_PR_AUC  Gap_ROC_AUC
1   top65_dedup52  lasso_features_top65--52.csv          52  ClassWeight           -       XGBoost          0.5176  0.443917     0.823789        0.303818     0.439283      0.959664 -0.006067   -0.161164       0.026947   -0.004106     0.000424
2   top50_dedup43  lasso_features_top50--43.csv          43  ClassWeight           -       XGBoost          0.5138  0.441964     0.872247        0.295964     0.413525      0.959078 -0.020723   -0.219702       0.018341   -0.001836    -0.001944
3   top55_dedup45  lasso_features_top55--45.csv          45  ClassWeight           -       XGBoost          0.5130  0.435165     0.872247        0.289898     0.415925      0.958711 -0.007

# LSTM

In [ ]:
# ============================================================
# 0. 라이브러리
# ============================================================

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import BorderlineSMOTE
from ctgan import CTGAN

from sklearn.metrics import (
    accuracy_score, roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score
)

import warnings
warnings.filterwarnings("ignore")


# ============================================================
# 1. 설정값
# ============================================================

TRAIN_PATH = r'10,11,12번\train데이터\M19_도매_소매업_train.parquet'
TEST_PATH  = r'10,11,12번\test데이터\M19_도매_소매업_test.parquet'

TARGET_COL   = "부실라벨_ICR3년"
RANDOM_STATE = 42
THRESHOLD    = 0.5
YEAR_COL     = "회계년도"
ID_COL       = "사업자등록번호"
ID_COLS      = ["회사명", "사업자등록번호", "회계년도"]

FOLD_VAL_YEARS = [2016, 2017, 2018, 2019, 2020, 2021]
TRAIN_START    = 2012
WINDOW_SIZES   = [2, 3, 5]

PRIMARY_METRICS = ["F1", "Recall", "ROC_AUC", "Precision", "PR_AUC", "Accuracy"]


# ============================================================
# 2. 피처 파일 (트리모델과 동일)
# ============================================================

feature_files = {
    # "top50_dedup43": r"13번.피처셀렉션\M19_도매_소매업\lasso_features_top50--43.csv",
    "top55_dedup45": r"13번.피처셀렉션\M19_도매_소매업\lasso_features_top55--45.csv",
    # "top60_dedup49": r"13번.피처셀렉션\M19_도매_소매업\lasso_features_top60--49.csv",
    # "top65_dedup52": r"13번.피처셀렉션\M19_도매_소매업\lasso_features_top65--52.csv",
}


# ============================================================
# 3. 데이터 로드
# ============================================================

train_full = pd.read_parquet(TRAIN_PATH)
test       = pd.read_parquet(TEST_PATH)

y_train_full = train_full[TARGET_COL]
y_test       = test[TARGET_COL]

print("=" * 70)
print(f"Train shape: {train_full.shape} | Test shape: {test.shape}")
print("=" * 70)


# ============================================================
# 4. 피처 목록 로드 & 유효성 확인
# ============================================================

feature_map = {}

for feature_name, feature_path in feature_files.items():
    df_feat        = pd.read_csv(feature_path)
    col_key        = "feature" if "feature" in df_feat.columns else df_feat.columns[0]
    raw_features   = df_feat[col_key].tolist()
    valid_features = [f for f in raw_features if f in train_full.columns]
    feature_map[feature_name] = valid_features
    print(f"[{feature_name}]  파일 내 피처: {len(raw_features)}개  "
          f"→ 실제 사용: {len(valid_features)}개")

print("=" * 70)


# ============================================================
# 5. LSTM 모델 정의
# ============================================================

class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze()


# ============================================================
# 6. 시계열 시퀀스 생성 함수
# ============================================================

def make_sequences(df, feature_cols, target_col, window=3):
    """
    기업별(ID_COL 기준) 슬라이딩 윈도우 시퀀스 생성.
    imputation이 완료된 df를 받아서 시퀀스로 변환.
    """
    X_list, y_list = [], []
    for _, group in df.groupby(ID_COL):
        group = group.sort_values(YEAR_COL).reset_index(drop=True)
        X = group[feature_cols].values.astype(np.float32)
        y = group[target_col].values.astype(np.float32)
        for i in range(len(group) - window + 1):
            X_list.append(X[i : i + window])
            y_list.append(y[i + window - 1])

    if len(X_list) == 0:
        return np.empty((0, window, len(feature_cols)), dtype=np.float32), \
               np.empty((0,), dtype=np.float32)

    return (
        np.array(X_list, dtype=np.float32),
        np.array(y_list, dtype=np.float32)
    )


# ============================================================
# 7. 오버샘플링 함수
# ============================================================

def apply_none(X_seq, y_seq, ratio=None):
    return X_seq.copy(), y_seq.copy()


def apply_borderline_smote(X_seq, y_seq, ratio):
    """Flatten → BorderlineSMOTE → Reshape"""
    n, w, f = X_seq.shape
    X_flat  = X_seq.reshape(n, w * f)

    smote = BorderlineSMOTE(
        sampling_strategy=ratio,
        random_state=RANDOM_STATE,
        kind="borderline-1"
    )
    X_res, y_res = smote.fit_resample(X_flat, y_seq)
    return X_res.reshape(-1, w, f).astype(np.float32), y_res.astype(np.float32)


def apply_ctgan(X_seq, y_seq, ratio):
    """마지막 시점 단면만 CTGAN 합성 → 시퀀스 복원"""
    w, f       = X_seq.shape[1], X_seq.shape[2]
    n_majority = int((y_seq == 0).sum())
    n_minority = int((y_seq == 1).sum())
    n_to_gen   = max(int(n_majority * ratio) - n_minority, 0)

    if n_to_gen == 0:
        return X_seq.copy(), y_seq.copy()

    minority_last = X_seq[y_seq == 1, -1, :]
    ctgan = CTGAN(epochs=100, verbose=False)
    ctgan.fit(pd.DataFrame(minority_last), discrete_columns=[])
    syn_last = ctgan.sample(n_to_gen).values.astype(np.float32)

    mean_prefix    = X_seq[y_seq == 1].mean(axis=0)
    syn_seqs       = np.tile(mean_prefix[np.newaxis], (n_to_gen, 1, 1))
    syn_seqs[:, -1, :] = syn_last

    return (
        np.concatenate([X_seq, syn_seqs], axis=0).astype(np.float32),
        np.concatenate([y_seq, np.ones(n_to_gen, dtype=np.float32)])
    )


# ============================================================
# 8. method_configs
# 튜플 구조: (오버샘플링 함수, ratio 리스트, pos_weight 사용 여부)
# ============================================================

method_configs = {
    # "None"            : (apply_none,               [None],       False),
    "ClassWeight"     : (apply_none,               [None],       True),
    # "BorderlineSMOTE" : (apply_borderline_smote,   [0.2, 0.1],  False),
    # "CTGAN"           : (apply_ctgan,              [0.2, 0.1],  False),
}


# ============================================================
# 9. 평가 지표 계산 헬퍼
# ============================================================

def calc_metrics(y_true, y_prob, threshold=THRESHOLD):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "F1"        : f1_score(y_true, y_pred, zero_division=0),
        "Recall"    : recall_score(y_true, y_pred, zero_division=0),
        "ROC_AUC"   : roc_auc_score(y_true, y_prob),
        "Precision" : precision_score(y_true, y_pred, zero_division=0),
        "PR_AUC"    : average_precision_score(y_true, y_prob),
        "Accuracy"  : accuracy_score(y_true, y_pred),
    }


# ============================================================
# 10. Expanding Window CV 인덱스 생성
# ============================================================

def make_expanding_folds(df, fold_val_years, train_start):
    folds = []
    for val_year in fold_val_years:
        train_idx = df.index[
            (df[YEAR_COL] >= train_start) & (df[YEAR_COL] < val_year)
        ]
        val_idx = df.index[df[YEAR_COL] == val_year]
        if len(train_idx) > 0 and len(val_idx) > 0:
            folds.append((train_idx, val_idx))
        else:
            print(f"  [경고] val_year={val_year} fold 생성 불가")
    return folds


# ============================================================
# 11. LSTM 학습 함수
# ============================================================

def train_lstm(X_seq_res, y_res, n_features,
               use_pos_weight=False, epochs=30, batch_size=64, lr=1e-3):

    X_tr = torch.tensor(X_seq_res, dtype=torch.float32)
    y_tr = torch.tensor(y_res,     dtype=torch.float32)

    loader = DataLoader(
        TensorDataset(X_tr, y_tr),
        batch_size=batch_size,
        shuffle=True
    )

    model     = LSTMClassifier(input_size=n_features)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    n_neg = (y_res == 0).sum()
    n_pos = (y_res == 1).sum()

    if use_pos_weight and n_pos > 0:
        criterion = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor([n_neg / n_pos])
        )
    else:
        criterion = nn.BCEWithLogitsLoss()

    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

    return model


def predict_lstm(model, X_seq):
    model.eval()
    with torch.no_grad():
        X_te   = torch.tensor(X_seq, dtype=torch.float32)
        y_prob = torch.sigmoid(model(X_te)).numpy()
    return y_prob


# ============================================================
# 12. 메인 실험 루프
# ============================================================

cv_results   = []
test_results = []

folds = make_expanding_folds(train_full, FOLD_VAL_YEARS, TRAIN_START)

for feature_name, use_features in feature_map.items():

    feature_file_name = feature_files[feature_name].split("\\")[-1]
    n_features        = len(use_features)

    print(f"\n{'='*70}")
    print(f"[Feature Set: {feature_name}]  피처 수: {n_features}개")
    print(f"{'='*70}")

    # ── NaN imputation: 전체 train fit → train/test transform ──
    # (시퀀스 생성 전에 먼저 처리해야 누수 없음)
    imputer     = SimpleImputer(strategy="median")
    train_imp   = train_full.copy()
    test_imp    = test.copy()

    train_imp[use_features] = imputer.fit_transform(train_full[use_features])
    test_imp[use_features]  = imputer.transform(test[use_features])

    for method_name, (oversample_fn, ratios, use_pos_weight) in method_configs.items():

        print(f"\n  {'='*60}")
        print(f"  방식: {method_name}")

        for ratio in ratios:
            ratio_label = ratio if ratio is not None else "-"
            print(f"\n    ratio = {ratio_label}")

            for window in WINDOW_SIZES:
                print(f"\n      window = {window}")

                # ── (A) Expanding Window CV ───────────────────────
                for fold_idx, (train_idx, val_idx) in enumerate(folds, start=1):

                    val_year = FOLD_VAL_YEARS[fold_idx - 1]

                    # fold별 imputation (누수 방지: fold train에만 fit)
                    imp_fold  = SimpleImputer(strategy="median")
                    fold_train_df = train_full.loc[train_idx].copy()
                    fold_val_df   = train_full.loc[val_idx].copy()

                    fold_train_df[use_features] = imp_fold.fit_transform(
                        train_full.loc[train_idx, use_features]
                    )
                    fold_val_df[use_features] = imp_fold.transform(
                        train_full.loc[val_idx, use_features]
                    )

                    X_seq_train, y_seq_train = make_sequences(
                        fold_train_df, use_features, TARGET_COL, window=window
                    )
                    X_seq_val, y_seq_val = make_sequences(
                        fold_val_df, use_features, TARGET_COL, window=window
                    )

                    # 부실 샘플 없으면 스킵
                    if (y_seq_train == 1).sum() == 0 or len(y_seq_val) == 0:
                        print(f"        [경고] fold {fold_idx} / window={window} "
                              f"→ 부실 샘플 없음, 스킵")
                        continue

                    # val에 부실이 없으면 ROC_AUC 계산 불가 → 스킵
                    if (y_seq_val == 1).sum() == 0:
                        print(f"        [경고] fold {fold_idx} val에 부실 없음, 스킵")
                        continue

                    X_res, y_res = oversample_fn(X_seq_train, y_seq_train, ratio)

                    n0 = int((y_res == 0).sum())
                    n1 = int((y_res == 1).sum())

                    model     = train_lstm(X_res, y_res, n_features,
                                           use_pos_weight=use_pos_weight)
                    y_prob_val = predict_lstm(model, X_seq_val)

                    val_metrics = calc_metrics(y_seq_val, y_prob_val)

                    cv_results.append({
                        "FeatureSet"  : feature_name,
                        "FeatureFile" : feature_file_name,
                        "N_Features"  : n_features,
                        "Method"      : method_name,
                        "SMOTE_Ratio" : ratio_label,
                        "Model"       : "LSTM",
                        "Window"      : window,
                        "Fold"        : fold_idx,
                        "Val_Year"    : val_year,
                        "Train_N0"    : n0,
                        "Train_N1"    : n1,
                        **{f"Val_{k}": v for k, v in val_metrics.items()},
                    })

                # ── (B) Test 평가: 전체 train 재학습 ─────────────
                X_seq_train_all, y_seq_train_all = make_sequences(
                    train_imp, use_features, TARGET_COL, window=window
                )
                X_seq_test_all, y_seq_test_all = make_sequences(
                    test_imp, use_features, TARGET_COL, window=window
                )

                if (y_seq_train_all == 1).sum() == 0 or len(y_seq_test_all) == 0:
                    print(f"      [경고] window={window} Test 부실 샘플 없음, 스킵")
                    continue

                X_tr_res, y_tr_res = oversample_fn(
                    X_seq_train_all, y_seq_train_all, ratio
                )

                final_model  = train_lstm(X_tr_res, y_tr_res, n_features,
                                          use_pos_weight=use_pos_weight)
                y_prob_test  = predict_lstm(final_model, X_seq_test_all)
                test_metrics = calc_metrics(y_seq_test_all, y_prob_test)

                test_results.append({
                    "FeatureSet"  : feature_name,
                    "FeatureFile" : feature_file_name,
                    "N_Features"  : n_features,
                    "Method"      : method_name,
                    "SMOTE_Ratio" : ratio_label,
                    "Model"       : "LSTM",
                    "Window"      : window,
                    **{f"Test_{k}": v for k, v in test_metrics.items()},
                })

                print(
                    f"        [LSTM window={window}]  "
                    f"Test F1={test_metrics['F1']:.4f} | "
                    f"Recall={test_metrics['Recall']:.4f} | "
                    f"PR_AUC={test_metrics['PR_AUC']:.4f}"
                )


# ============================================================
# 13. 결과 DataFrame 변환
# ============================================================

cv_df   = pd.DataFrame(cv_results)
test_df = pd.DataFrame(test_results)


# ============================================================
# 14. CV 평균 집계 + Test 결과 병합 → summary_df
# ============================================================

GROUP_KEYS = ["FeatureSet", "FeatureFile", "N_Features",
              "Method", "SMOTE_Ratio", "Model", "Window"]

cv_agg = (
    cv_df
    .groupby(GROUP_KEYS)
    [[f"Val_{m}" for m in PRIMARY_METRICS]]
    .mean()
    .reset_index()
)

summary_df = cv_agg.merge(
    test_df,
    on=GROUP_KEYS,
    how="left"
)

for m in PRIMARY_METRICS:
    summary_df[f"Gap_{m}"] = summary_df[f"Val_{m}"] - summary_df[f"Test_{m}"]

print(f"\nsummary_df shape: {summary_df.shape}")
print(summary_df[["Method", "Window", "Val_F1", "Test_F1", "Gap_F1"]].to_string(index=False))


# ============================================================
# 15. 저장
# ============================================================

summary_df.to_csv(
    "Summary_LSTM_CV_Test_모델별_파일별_불균형방식별.csv",
    index=False, encoding="utf-8-sig"
)
cv_df.to_csv(
    "LSTM_CV_fold_results.csv",
    index=False, encoding="utf-8-sig"
)

print("\n" + "=" * 70)
print("저장 완료")
print(f"  → Summary_LSTM_CV_Test_모델별_파일별_불균형방식별.csv  ({len(summary_df)}행)")
print(f"  → LSTM_CV_fold_results.csv  ({len(cv_df)}행)")
print("=" * 70)